# Importar ficheiros CRONO (cargas) para DuckDB

1. Preparar tabelas `cargas_2025` / `cargas_2026`
2. Ler ficheiros CRONO da pasta
3. Extrair o ano dos dados e inserir na tabela correta
4. Controlar duplicados e validar a importação


In [ ]:
# ==========================
# 1. Imports e configuração
# ==========================

import csv
import platform
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd

if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\Documents\python\00.DB\2026.duckdb"
    )
    PASTA_FICHEIROS = Path(
        r"C:\Users\LISARR\Desktop\02.Pontualidade\crono"
    )

elif platform.system() == "Darwin":
    DB_PATH = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/00_DB/2026.duckdb"
    )
    PASTA_FICHEIROS = (
        Path.home()
        / "Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/2026_dados"
    )

else:
    raise OSError(
        f"Sistema operativo não suportado: {platform.system()}"
    )

DB_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"DB_PATH: {DB_PATH}")
print(f"PASTA_FICHEIROS: {PASTA_FICHEIROS}")


In [ ]:
# 49 colunas, exatamente como aparecem no ficheiro
COLUNAS_CARGAS_ORDENADAS = [
    ("TIPO_SUMINISTRO", "Tipo Suministro"),
    ("BASE", "Base"),
    ("LANZADERA", "Lanzadera"),
    ("PUNTO_SUMINISTRO", "Punto Suministro"),
    ("RUTA", "Ruta"),
    ("PALES_TMS", "Palés TMS"),
    ("TEMPERATURA_REQUERIDA", "Temperatura"),
    ("AGENCIA", "Agencia"),
    ("TRANSPORTISTA", "Transportista"),
    ("DNI", "DNI"),
    ("PRECINTO", "Precinto"),
    ("TELEFONO", "Teléfono"),
    ("TRACTORA", "Tractora"),
    ("REMOLQUE", "Remolque"),
    ("FECHA_PREVISTA_POSICIONAMIENTO", "Fecha Prevista Posicionamiento"),
    ("HORA_PREVISTA_POSICIONAMIENTO", "Hora Prevista Posicionamiento"),
    ("FECHA_REAL_POSICIONAMIENTO", "Fecha Real Posicionamiento"),
    ("HORA_REAL_POSICIONAMIENTO", "Hora Real Posicionamiento"),
    ("FECHA_REAL_ENTRADA", "Fecha Real Entrada"),
    ("HORA_REAL_ENTRADA", "Hora Real Entrada"),
    ("MUELLE", "Muelle"),
    ("FECHA_PREVISTA_SALIDA", "Fecha Prevista Salida"),
    ("HORA_PREVISTA_SALIDA", "Hora Prevista Salida"),
    ("FECHA_REAL_SALIDA", "Fecha Real Salida"),
    ("HORA_REAL_SALIDA", "Hora Real Salida"),
    ("FECHA_PREVISTA_ENTREGA", "Fecha Prevista Entrega"),
    ("HORA_PREVISTA_ENTREGA", "Hora Prevista Entrega"),
    ("FECHA_REAL_ENTREGA", "Fecha Real Entrega"),
    ("HORA_REAL_ENTREGA", "Hora Real Entrega"),
    ("HORA_SALIDA_ENTREGA", "Hora Salida Entrega"),
    ("OBSERVACIONES", "Observaciones"),
    ("COMENTARIOS", "Comentarios"),
    ("ZONA", "Zona"),
    ("CLIENTE", "Cliente"),
    ("AUTORIZADO_AUTOCARGA", "Autorizado autocarga"),
    ("AUTOCARGA", "Autocarga"),
    ("HUECOS_TMS", "Huecos TMS"),
    ("HUECOS_CARGA", "Huecos carga"),
    ("HUECOS_DESCARGA", "Huecos descarga"),
    ("TEMPERATURA_MEDIDA1", "Temperatura"),
    ("TEMPERATURA_MEDIDA2", "Temperatura2"),
    ("MOTIVO", "Motivo"),
    ("ESTADO", "Estado"),
    ("ESTADO_MERCANCIA", "Estado mercancía"),
    ("ESTADO_CAJA", "Estado caja"),
    ("ESTADO_OLORES", "Estado olores"),
    ("ESTADO_LIMPIEZA_VEHICULO", "Estado limpieza vehículo"),
    ("ESTADO_VEHICULO_SECO", "Estado vehículo seco"),
    ("ESTADO_LIBRE_PLAGAS", "Estado libre de plagas"),
]

COLUNAS_CARGAS = {nome_sql: nome_ficheiro for nome_sql, nome_ficheiro in COLUNAS_CARGAS_ORDENADAS}
ORDEM_CABECALHO_CARGAS = [nome_ficheiro for _, nome_ficheiro in COLUNAS_CARGAS_ORDENADAS]

COLUNA_ORIGEM = "ficheiro_origem"
ANOS = ("2025", "2026")
CHAVE_UNICA_CARGAS = tuple(COLUNAS_CARGAS.keys())

print(f"Colunas definidas: {len(COLUNAS_CARGAS)}")

In [ ]:
# ==========================
# 3. Estrutura DuckDB
# ==========================

def preparar_base_dados(con):
    colunas_sql = ",\n        ".join(
        f'"{coluna}" VARCHAR'
        for coluna in COLUNAS_CARGAS.keys()
    )

    for ano in ANOS:
        tabela = f"cargas_{ano}"

        con.execute(f"""
            CREATE TABLE IF NOT EXISTS "{tabela}" (
                id BIGINT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" VARCHAR
            )
        """)

        con.execute(
            f'CREATE UNIQUE INDEX IF NOT EXISTS '
            f'"idx_{tabela}_chave" '
            f'ON "{tabela}" '
            f'({", ".join(f""""{c}"""" for c in CHAVE_UNICA_CARGAS)})'
        )


with duckdb.connect(str(DB_PATH)) as con:
    preparar_base_dados(con)

print("Base de dados, tabelas e índices prontos.")


In [ ]:
# ==========================
# 4. Leitura dos ficheiros
# ==========================

def listar_ficheiros_excel(pasta):
    ficheiros = {
        caminho.resolve()
        for caminho in Path(pasta).rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower() == ".xls"
        )
    }

    return sorted(ficheiros)


def validar_data(valor):
    if not valor:
        return None

    texto = str(valor).strip()

    if (
        not texto
        or texto.upper() == "N/D"
    ):
        return None

    try:
        return datetime.strptime(
            texto,
            "%d/%m/%Y",
        )

    except ValueError:
        return None


def ler_linhas_csv_cargas(caminho_ficheiro):
    with caminho_ficheiro.open(
        "r",
        encoding="iso-8859-1",
        newline="",
    ) as ficheiro:

        linhas_ficheiro = list(
            csv.reader(
                ficheiro,
                delimiter="\t",
            )
        )

    if not linhas_ficheiro:
        return [], []

    cabecalho = linhas_ficheiro[0]
    linhas = linhas_ficheiro[1:]

    n_colunas = len(cabecalho)
    linhas_normalizadas = []

    for valores in linhas:

        if len(valores) < n_colunas:
            valores = (
                valores
                + [None] * (n_colunas - len(valores))
            )

        elif len(valores) > n_colunas:
            valores = valores[:n_colunas]

        linhas_normalizadas.append(
            valores
        )

    return cabecalho, linhas_normalizadas


ficheiros = listar_ficheiros_excel(
    PASTA_FICHEIROS
)

print(
    f"Ficheiros encontrados: "
    f"{len(ficheiros)}"
)


In [ ]:
# ==========================
# 5. Funções de preparação
# ==========================

def extrair_ano_e_data_cargas(valores):
    colunas_ordem = list(
        COLUNAS_CARGAS.keys()
    )

    try:
        indice = colunas_ordem.index(
            "FECHA_PREVISTA_POSICIONAMIENTO"
        )

        if indice < len(valores):

            data_str = valores[indice]
            data_obj = validar_data(
                data_str
            )

            if data_obj:
                return (
                    str(data_obj.year),
                    data_str,
                )

    except (ValueError, IndexError):
        pass

    return None, None


def montar_linha_cargas(
    valores,
    nome_ficheiro,
):
    colunas_ordem = list(
        COLUNAS_CARGAS.keys()
    )

    return (
        tuple(
            valores[:len(colunas_ordem)]
        )
        + (nome_ficheiro,)
    )


def inserir_batch(
    con,
    ano,
    linhas,
):
    if not linhas:
        return 0, 0

    tabela = f"cargas_{ano}"

    antes = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    max_id = con.execute(f"""
        SELECT COALESCE(
            MAX(TRY_CAST(id AS BIGINT)),
            0
        )
        FROM "{tabela}"
    """).fetchone()[0]

    colunas = [
        *COLUNAS_CARGAS.keys(),
        COLUNA_ORIGEM,
    ]

    df_batch = pd.DataFrame(
        linhas,
        columns=colunas,
        dtype=object,
    )

    df_batch.insert(
        0,
        "id",
        range(
            int(max_id) + 1,
            int(max_id) + 1 + len(df_batch),
        ),
    )

    con.register(
        "batch_cargas",
        df_batch,
    )

    try:
        con.execute(f"""
            INSERT OR IGNORE INTO "{tabela}"
            SELECT *
            FROM batch_cargas
        """)

    finally:
        con.unregister(
            "batch_cargas"
        )

    depois = con.execute(
        f'SELECT COUNT(*) FROM "{tabela}"'
    ).fetchone()[0]

    inseridos = depois - antes
    duplicados = len(linhas) - inseridos

    return inseridos, duplicados


print("Funções prontas.")


In [ ]:
# ==========================
# 6. Importação
# ==========================

contagem_novas = {
    ano: 0
    for ano in ANOS
}

total_duplicadas = 0
total_sem_data = 0
total_sem_bd = 0
total_processados = 0
total_cabecalho_invalido = 0
total_erros = 0

linhas_sem_bd = []
linhas_com_erro = []

with duckdb.connect(str(DB_PATH)) as con:

    tabelas_existentes = {
        linha[0]
        for linha in con.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'main'
              AND table_name LIKE 'cargas_%'
        """).fetchall()
    }

    anos_disponiveis_str = {
        tabela.split("_", 1)[1]
        for tabela in tabelas_existentes
        if tabela.startswith("cargas_")
    }

    print(
        "Bases disponíveis para anos: "
        f"{sorted(anos_disponiveis_str)}\n"
    )

    for numero, caminho in enumerate(
        ficheiros,
        start=1,
    ):
        nome_ficheiro = caminho.name
        transacao_aberta = False

        try:
            cabecalho, linhas = (
                ler_linhas_csv_cargas(
                    caminho
                )
            )

            if (
                cabecalho
                != ORDEM_CABECALHO_CARGAS
            ):
                total_cabecalho_invalido += 1

                print(
                    f"[{numero}/{len(ficheiros)}] "
                    f"[AVISO] cabeçalho de "
                    f"'{nome_ficheiro}' diferente "
                    f"— ficheiro ignorado."
                )

                continue

            novas_ficheiro = {
                ano: 0
                for ano in ANOS
            }

            duplicadas_ficheiro = 0
            sem_data_ficheiro = 0
            sem_bd_ficheiro = 0
            erro_ficheiro = 0

            batches = {
                ano: []
                for ano in ANOS
            }

            for indice_linha, valores in enumerate(
                linhas,
                start=2,
            ):
                if all(
                    valor is None
                    or str(valor).strip() == ""
                    for valor in valores
                ):
                    continue

                ano, _ = (
                    extrair_ano_e_data_cargas(
                        valores
                    )
                )

                if ano not in ANOS:
                    sem_data_ficheiro += 1
                    continue

                linha = montar_linha_cargas(
                    valores,
                    nome_ficheiro,
                )

                if ano not in anos_disponiveis_str:
                    sem_bd_ficheiro += 1
                    total_sem_bd += 1

                    linhas_sem_bd.append(
                        (
                            ano,
                            linha,
                            indice_linha,
                        )
                    )

                    continue

                batches[ano].append(
                    linha
                )

            con.begin()
            transacao_aberta = True

            for ano in ANOS:

                try:
                    inseridos, duplicados = (
                        inserir_batch(
                            con,
                            ano,
                            batches[ano],
                        )
                    )

                    novas_ficheiro[ano] += (
                        inseridos
                    )

                    duplicadas_ficheiro += (
                        duplicados
                    )

                except Exception as erro:
                    erro_ficheiro += 1

                    linhas_com_erro.append(
                        (
                            ano,
                            str(erro),
                            None,
                        )
                    )

                    raise

            con.commit()
            transacao_aberta = False

            total_processados += 1

            for ano in ANOS:
                contagem_novas[ano] += (
                    novas_ficheiro[ano]
                )

            total_duplicadas += (
                duplicadas_ficheiro
            )

            total_sem_data += (
                sem_data_ficheiro
            )

            partes = []

            for ano in ANOS:
                if novas_ficheiro[ano] > 0:
                    partes.append(
                        f"{novas_ficheiro[ano]} "
                        f"novas {ano}"
                    )

            if duplicadas_ficheiro > 0:
                partes.append(
                    f"{duplicadas_ficheiro} dup"
                )

            if sem_bd_ficheiro > 0:
                partes.append(
                    f"[SEM BD] "
                    f"{sem_bd_ficheiro}"
                )

            if sem_data_ficheiro > 0:
                partes.append(
                    f"{sem_data_ficheiro} "
                    f"sem data"
                )

            if erro_ficheiro > 0:
                partes.append(
                    f"[ERRO] {erro_ficheiro}"
                )

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"{nome_ficheiro} -> "
                + (
                    " | ".join(partes)
                    if partes
                    else "nenhuma"
                )
            )

        except Exception as erro:

            if transacao_aberta:
                con.rollback()

            total_erros += 1

            print(
                f"[{numero}/{len(ficheiros)}] "
                f"[ERRO] '{nome_ficheiro}': "
                f"{erro}"
            )

    con.checkpoint()


print("\n" + "=" * 70)

print(
    f"2025: {contagem_novas['2025']} novas | "
    f"2026: {contagem_novas['2026']} novas | "
    f"Dup: {total_duplicadas} | "
    f"Sem BD: {total_sem_bd}"
)

print("=" * 70)


In [ ]:
# ==========================
# 7. Validação final
# ==========================

with duckdb.connect(
    str(DB_PATH),
    read_only=True,
) as con:

    validacao = []
    duplicados = []

    for ano in ANOS:

        tabela = f"cargas_{ano}"

        existe = con.execute("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_schema = 'main'
              AND table_name = ?
        """, [tabela]).fetchone()[0]

        if not existe:
            continue

        total, ids_unicos = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(DISTINCT id)
            FROM "{tabela}"
        """).fetchone()

        grupos_duplicados, linhas_duplicadas = con.execute(f"""
            SELECT
                COUNT(*),
                COALESCE(SUM(n), 0)
            FROM (
                SELECT
                    {", ".join(f'"{c}"' for c in CHAVE_UNICA_CARGAS)},
                    COUNT(*) AS n
                FROM "{tabela}"
                GROUP BY
                    {", ".join(f'"{c}"' for c in CHAVE_UNICA_CARGAS)}
                HAVING COUNT(*) > 1
            )
        """).fetchone()

        validacao.append({
            "Tabela": tabela,
            "Linhas": total,
            "IDs_Unicos": ids_unicos,
            "Grupos_Duplicados": grupos_duplicados,
            "Linhas_Duplicadas": linhas_duplicadas,
            "OK": (
                total == ids_unicos
                and grupos_duplicados == 0
            ),
        })

        if grupos_duplicados > 0:

            condicao = " AND ".join(
                f't."{coluna}" '
                f'IS NOT DISTINCT FROM '
                f'd."{coluna}"'
                for coluna
                in CHAVE_UNICA_CARGAS
            )

            df_dup = con.execute(f"""
                WITH chaves_duplicadas AS (
                    SELECT
                        {", ".join(f'"{c}"' for c in CHAVE_UNICA_CARGAS)},
                        COUNT(*) AS qtd_duplicados
                    FROM "{tabela}"
                    GROUP BY
                        {", ".join(f'"{c}"' for c in CHAVE_UNICA_CARGAS)}
                    HAVING COUNT(*) > 1
                )

                SELECT
                    d.qtd_duplicados,
                    t.*
                FROM "{tabela}" t
                INNER JOIN chaves_duplicadas d
                    ON {condicao}
                ORDER BY
                    TRY_CAST(t.id AS BIGINT)
            """).df()

            df_dup.insert(
                0,
                "Tabela",
                tabela,
            )

            duplicados.append(
                df_dup
            )

df_validacao = pd.DataFrame(
    validacao
)

df_duplicados = (
    pd.concat(
        duplicados,
        ignore_index=True,
    )
    if duplicados
    else pd.DataFrame()
)

df_validacao
